In [1]:
from hashlib import blake2b, sha256, sha3_256
from bitarray import bitarray


class BloomFilter:

  def __init__(self, size: int, hash_count: int = 3):
    self.size = size
    self.hash_count = hash_count
    self.bit_array = bitarray(size)
    self.bit_array.setall(0)

  def _get_hashes(self, item: str):
    s = item.lower().encode()
    hashes = [
        int(sha256(s).hexdigest(), 16) % self.size,
        int(blake2b(s).hexdigest(), 16) % self.size,
        int(sha3_256(s).hexdigest(), 16) % self.size,
    ]
    return hashes[: self.hash_count]

  def add(self, item: str):
    for h in self._get_hashes(item):
      self.bit_array[h] = True

  def __contains__(self, item: str) -> bool:
    return all(self.bit_array[h] for h in self._get_hashes(item))

In [2]:
with open('words.txt') as f:
  words = [line.strip().lower() for line in f if line.strip()]

In [3]:
bf1 = BloomFilter(size=10**7, hash_count=1)
bf2 = BloomFilter(size=10**7, hash_count=2)
bf3 = BloomFilter(size=10**7, hash_count=3)

for word in words:
    bf1.add(word)
    bf2.add(word)
    bf3.add(word)

In [4]:
import string

def suggest_corrections(
    typed_word: str,
    bloom_filter: BloomFilter
) -> list[str]:
    word = typed_word.lower()
    suggestions = []

    for i in range(len(word)):
        for letter in string.ascii_lowercase:
            if letter == word[i]:
                continue

            candidate = word[:i] + letter + word[i + 1:]

            if candidate in bloom_filter:
                suggestions.append(candidate)

    return suggestions

In [8]:
print(suggest_corrections("floeer", bf1))
print(suggest_corrections("floeer", bf2))
print(suggest_corrections("floeer", bf3))

['bloeer', 'qloeer', 'fyoeer', 'flofer', 'floter', 'flower', 'floeqr', 'floees']
['fyoeer', 'floter', 'flower']
['floter', 'flower']
